In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoModel
from PIL import Image
import os
from pathlib import Path
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip -q "/content/drive/MyDrive/mitosis_detector_BTTAI/dataset_root.zip" -d "/content/dataset_root"

replace /content/dataset_root/dataset_root/1/Copy of 8_19.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
class Config:
    data_dir = "/content/dataset_root/dataset_root"

    # Training parameters
    batch_size = 256
    num_epochs = 20 # should increase
    learning_rate = 1e-4 * (batch_size / 8)  # = 2e-4
    train_split = 0.8

    # Early stopping
    patience = 10

    # Model parameters
    model_name = "kaiko-ai/midnight"
    embedding_dim = 1536
    num_classes = 2

    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Random seed
    seed = 123

In [ ]:
# set seed
torch.manual_seed(Config.seed)
np.random.seed(Config.seed)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

val_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])


In [ ]:
# how to load and process the images
# gets an image, converts to RGB, applies trasnformations and normalizations and returns tensor embeddings
class MitoticDataset(Dataset):
  def __init__(self, image_paths, labels, transform=None):
    self.image_paths = image_paths  # Fixed: was image_path (missing 's')
    self.labels = labels
    self.transform = transform  # Fixed: was transfrom (typo)

  def __len__(self):  # Fixed: indentation - should be at class level, not inside __init__
    return len(self.image_paths)

  def __getitem__(self, idx):  # Fixed: indentation - should be at class level
    image = Image.open(self.image_paths[idx]).convert('RGB')  # Fixed: image_paths
    label = self.labels[idx]  # Fixed: was self.label (missing 's')

    if self.transform:
        image = self.transform(image)

    return image, label

In [ ]:
# data preperation
def load_data(data_dir, train_split=0.8):
    """Load images from folders and create train/val splits"""
    image_paths = []
    labels = []

    # maps class names to numric labels
    class_names = ['1', '2']
    class_to_idx = {'1': 0, '2': 1}

    # Load images from each folder
    for class_name in class_names:

        class_folder = Path(data_dir) / class_name
        if not class_folder.exists():
            raise ValueError(f"Folder not found: {class_folder}")

        class_images = list(class_folder.glob('*.jpeg'))
        print(f"Loading {len(class_images)} images from {class_name}/")

        for img_path in class_images:
            image_paths.append(str(img_path))
            labels.append(class_to_idx[class_name])

    # Convert to numpy arrays
    image_paths = np.array(image_paths)
    labels = np.array(labels)

    # Shuffle because we don't want model to recognize any particular order or take into account the ordering pattern when training
    indices = np.random.permutation(len(image_paths))
    image_paths = image_paths[indices]
    labels = labels[indices]

    # Split into train and validation [TODO: also try cross validation]
    split_idx = int(len(image_paths) * train_split)

    train_paths = image_paths[:split_idx]
    train_labels = labels[:split_idx]

    val_paths = image_paths[split_idx:]
    val_labels = labels[split_idx:]

    print(f"Total images: {len(image_paths)}")
    print(f"Training: {len(train_paths)} images")
    print(f"Validation: {len(val_paths)} images")
    print(f"Class distribution - Non-mitotic: {np.sum(labels == 0)}, Mitotic: {np.sum(labels == 1)}")

    return train_paths, train_labels, val_paths, val_labels, class_names

In [ ]:
# model
class MitoticClassifier(nn.Module):
    def __init__(self, model_name, num_classes, freeze_backbone=True, unfreeze_last_n=0):
        super(MitoticClassifier, self).__init__()

        # Load pretrained midnight model
        self.backbone = AutoModel.from_pretrained(model_name)

        # STEP 7: Selective freezing strategy
        if freeze_backbone:
            # Freeze everything
            for param in self.backbone.parameters():
                param.requires_grad = False
            print("Backbone: Fully frozen")
        else:
            # Freeze all first
            for param in self.backbone.parameters():
                param.requires_grad = False

            # Then unfreeze last N transformer layers
            if unfreeze_last_n > 0:
                print(f"Backbone: Unfreezing last {unfreeze_last_n} transformer layers")
                for i in range(-unfreeze_last_n, 0):
                    for param in self.backbone.encoder.layer[i].parameters():
                        param.requires_grad = True

        # STEP 3: Improved classifier head with BatchNorm and more capacity
        self.classifier = nn.Sequential(
            nn.Linear(Config.embedding_dim * 2, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, num_classes)
        )
def extract_features(self, x):

    """Extract features using Midnight's recommended approach"""
    # Use torch.no_grad() when backbone is frozen, otherwise allow gradients
    if self._is_backbone_frozen():
      with torch.no_grad():
        outputs = self.backbone(x)
    else:
      outputs = self.backbone(x)

    hidden_states = outputs.last_hidden_state

    # Extract CLS token and patch tokens
    cls_token = hidden_states[:, 0, :]  # [batch, embedding_dim]
    patch_tokens = hidden_states[:, 1:, :]  # [batch, num_patches, embedding_dim]

    # Average pool patch tokens
    patch_mean = patch_tokens.mean(dim=1)  # [batch, embedding_dim]

    # Concatenate CLS and patch mean
    features = torch.cat([cls_token, patch_mean], dim=-1)  # [batch, embedding_dim*2]

    return features
def _is_backbone_frozen(self):
    return not next(self.backbone.parameters()).requires_grad

def forward(self, x):
    features = self.extract_features(x)
    logits = self.classifier(features)
    return logits
    class_counts = np.bincount(train_labels)
print(f"Class counts: {class_counts}")  # Will show [12001, 14349]

# Create weights inversely proportional to class frequency
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * len(class_counts)
print(f"Class weights: {class_weights}")

# Move weights to device
class_weights = class_weights.to(Config.device)

# Use weighted CrossEntropyLoss instead of plain one
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)


In [ ]:
# training
def train_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  running_loss = 0.0
  all_preds = []
  all_labels = []

  for images, labels in tqdm(dataloader, desc = "Training"):
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    preds = torch.argmax(outputs, dim = 1)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

  epoch_loss = running_loss / len(dataloader)
  epoch_acc = accuracy_score(all_labels, all_preds)

  return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
  model.eval()
  running_loss = 0.0
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for images, labels in tqdm(dataloader, desc = "Validation"):
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)

      running_loss += loss.item()
      preds = torch.argmax(outputs, dim = 1)
      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  epoch_loss = running_loss/len(dataloader)
  epoch_acc = accuracy_score(all_labels, all_preds)

  return epoch_loss, epoch_acc, all_preds, all_labels

In [ ]:

# main training loop
def main():
    print("="*80)
    print("IMPROVED MITOSIS DETECTION TRAINING")
    print("="*80)
    print(f"Using device: {Config.device}")

    # Set seeds for reproducibility
    torch.manual_seed(Config.seed)
    np.random.seed(Config.seed)

    # Load data
    train_paths, train_labels, val_paths, val_labels, class_names = load_data(
        Config.data_dir, Config.train_split
    )

    # STEP 2: Calculate class weights for balanced loss
    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    class_weights = class_weights.to(Config.device)
    print(f"\nClass weights for loss: {class_weights}")

    # STEP 1: Create datasets with DIFFERENT transforms for train/val
    train_dataset = MitoticDataset(train_paths, train_labels, transform=train_transform)
    val_dataset = MitoticDataset(val_paths, val_labels, transform=val_transform)

    # Create weighted sampler for balanced batches
    samples_weight = class_weights[train_labels].cpu()
    sampler = WeightedRandomSampler(
        weights=samples_weight,
        num_samples=len(samples_weight),
        replacement=True
    )

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.batch_size,
        sampler=sampler,
        num_workers=4,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Initialize model
    print("\n" + "="*80)
    print("MODEL INITIALIZATION")
    print("="*80)
    model = MitoticClassifier(
        model_name=Config.model_name,
        num_classes=Config.num_classes,
        freeze_backbone=Config.freeze_backbone,
        unfreeze_last_n=Config.unfreeze_last_n_layers
    )
    model = model.to(Config.device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {total_params - trainable_params:,}")

    # STEP 2: Loss function with class weights and label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    # STEP 7: Optimizer with different learning rates for backbone and classifier
    if Config.freeze_backbone:
        optimizer = optim.AdamW(
            model.classifier.parameters(),
            lr=Config.base_learning_rate,
            weight_decay=0.01
        )
        print(f"Optimizer: AdamW with single LR = {Config.base_learning_rate}")
    else:
        optimizer = optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': Config.base_learning_rate * 0.1},
            {'params': model.classifier.parameters(), 'lr': Config.base_learning_rate}
        ], weight_decay=0.01)
        print(f"Optimizer: AdamW with backbone LR = {Config.base_learning_rate * 0.1}")
        print(f"           and classifier LR = {Config.base_learning_rate}")

    # STEP 4: OneCycleLR scheduler for better convergence
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=Config.base_learning_rate,
        epochs=Config.num_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos',
        div_factor=25,
        final_div_factor=1000
    )

    # STEP 5: Training loop with early stopping
    best_val_acc = 0.0
    best_val_loss = float('inf')
    patience_counter = 0

    print("\n" + "="*80)
    print("TRAINING START")
    print("="*80)

    for epoch in range(Config.num_epochs):
        print(f"\nEpoch {epoch+1}/{Config.num_epochs}")
        print("-" * 40)

        # Training
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, Config.device
        )

        # Validation
        val_loss, val_acc, val_preds, val_labels = validate(
            model, val_loader, criterion, Config.device
        )

        # Print metrics
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
        print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")

        # STEP 5: Early stopping logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model_improved.pth")
            print(f"✓ NEW BEST! Saved model (val acc: {val_acc:.4f})")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{Config.patience}")

            if patience_counter >= Config.patience:
                print(f"\n⚠ Early stopping triggered at epoch {epoch+1}")
                print(f"Best validation accuracy: {best_val_acc:.4f}")
                break

    # Final evaluation
    print("\n" + "="*80)
    print("FINAL EVALUATION")
    print("="*80)

    model.load_state_dict(torch.load("best_model_improved.pth"))
    _, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, Config.device)

    print(f"\n🎯 Best Validation Accuracy: {val_acc:.4f}")
    print(f"Best Validation Loss: {best_val_loss:.4f}")

    print("\n" + "="*80)
    print("CONFUSION MATRIX")
    print("="*80)
    cm = confusion_matrix(val_labels, val_preds)
    print(cm)

    # Calculate per-class metrics
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn)  # Recall for mitotic class
    specificity = tn / (tn + fp)  # Recall for non-mitotic class
    precision = tp / (tp + fp)

    print(f"\nDetailed Metrics:")
    print(f"  Sensitivity (Mitotic Recall):     {sensitivity:.4f}")
    print(f"  Specificity (Non-mitotic Recall): {specificity:.4f}")
    print(f"  Precision (Mitotic):              {precision:.4f}")

    print("\n" + "="*80)
    print("CLASSIFICATION REPORT")
    print("="*80)
    print(classification_report(val_labels, val_preds, target_names=class_names))

    print("\n" + "="*80)
    print("TRAINING COMPLETE!")
    print("="*80)
    print(f"Model saved as: best_model_improved.pth")

In [ ]:
if __name__ == "__main__":
  main()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoModel
from PIL import Image
import os
from pathlib import Path
import numpy as np
from tqdm import tqdm

class Config:
    data_dir = "/content/dataset_root/dataset_root"
    batch_size = 256
    num_epochs = 20
    learning_rate = 1e-4 * (batch_size / 8)
    train_split = 0.8
    patience = 10
    model_name = "kaiko-ai/midnight"
    embedding_dim = 1536
    num_classes = 2
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    seed = 123

class MitoticDataset(Dataset):
  def __init__(self, image_paths, labels, transform=None):
    self.image_paths = image_paths
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    image = Image.open(self.image_paths[idx]).convert('RGB')
    label = self.labels[idx]
    if self.transform:
        image = self.transform(image)
    return image, label

def load_data(data_dir, train_split=0.8):
    """Load images from folders and create train/val splits"""
    image_paths = []
    labels = []

    class_names = ['1', '2']
    class_to_idx = {'1': 0, '2': 1}

    for class_name in class_names:
        class_folder = Path(data_dir) / class_name
        if not class_folder.exists():
            raise ValueError(f"Folder not found: {class_folder}")

        class_images = list(class_folder.glob('*.jpeg'))
        print(f"Loading {len(class_images)} images from {class_name}/")

        for img_path in class_images:
            image_paths.append(str(img_path))
            labels.append(class_to_idx[class_name])

    image_paths = np.array(image_paths)
    labels = np.array(labels)

    indices = np.random.permutation(len(image_paths))
    image_paths = image_paths[indices]
    labels = labels[indices]

    split_idx = int(len(image_paths) * train_split)

    train_paths = image_paths[:split_idx]
    train_labels = labels[:split_idx]

    val_paths = image_paths[split_idx:]
    val_labels = labels[split_idx:]

    print(f"Total images: {len(image_paths)}")
    print(f"Training: {len(train_paths)} images")
    print(f"Validation: {len(val_paths)} images")
    print(f"Class distribution - Non-mitotic: {np.sum(labels == 0)}, Mitotic: {np.sum(labels == 1)}")

    return train_paths, train_labels, val_paths, val_labels, class_names

class MitoticClassifier(nn.Module):
  def __init__(self, model_name, num_classes, freeze_backbone = True):
    super(MitoticClassifier, self).__init__()

    self.backbone = AutoModel.from_pretrained(model_name)

    if freeze_backbone:
      for param in self.backbone.parameters():
        param.requires_grad = False

    self.classifier = nn.Sequential(
        nn.Linear(Config.embedding_dim * 2, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )

  def extract_features(self, x):
    """Extract features using Midnight's recommended approach"""
    if self._is_backbone_frozen():
      with torch.no_grad():
        outputs = self.backbone(x)
    else:
      outputs = self.backbone(x)

    hidden_states = outputs.last_hidden_state


    cls_token = hidden_states[:, 0, :]
    patch_tokens = hidden_states[:, 1:, :]

    patch_mean = patch_tokens.mean(dim=1)

    features = torch.cat([cls_token, patch_mean], dim=-1)

    return features

  def _is_backbone_frozen(self):
    return not next(self.backbone.parameters()).requires_grad

  def forward(self, x):
    features = self.extract_features(x)
    logits = self.classifier(features)
    return logits

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

def extract_and_save(loader, save_dir, split_name):
    all_features = []
    all_labels = []

    print(f"Extracting features for {split_name}...")
    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Extracting {split_name}"):
            images = images.to(Config.device)

            features = model.extract_features(images)

            all_features.append(features.cpu())
            all_labels.append(labels.cpu())

    final_features = torch.cat(all_features, dim=0)
    final_labels = torch.cat(all_labels, dim=0)

    print(f"Finished. Feature tensor shape: {final_features.shape}")
    print(f"Finished. Label tensor shape: {final_labels.shape}")

    torch.save(final_features, os.path.join(save_dir, f"{split_name}_features.pt"))
    torch.save(final_labels, os.path.join(save_dir, f"{split_name}_labels.pt"))

if __name__ == "__main__":
    print(f"Using device: {Config.device}")
    torch.manual_seed(Config.seed)
    np.random.seed(Config.seed)

    os.makedirs("features", exist_ok=True)

    # Load data paths
    train_paths, train_labels, val_paths, val_labels, _ = load_data(Config.data_dir, Config.train_split)

    # Create datasets
    train_dataset = MitoticDataset(train_paths, train_labels, transform=transform)
    val_dataset = MitoticDataset(val_paths, val_labels, transform=transform)

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Load Model
    print("Loading Midnight-12k model for feature extraction...")
    model = MitoticClassifier(
        model_name=Config.model_name,
        num_classes=Config.num_classes,
        freeze_backbone=True
    )
    model = model.to(Config.device)
    model.eval() # Set model to evaluation mode

    # Run the extraction for both train and validation sets
    extract_and_save(train_loader, "features", "train")
    extract_and_save(val_loader, "features", "val")

    print("\nFeature extraction complete!")


In [ ]:
# Fast Training
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np
from tqdm import tqdm
import os
import glob

class Config:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    TRAIN_FEATURES_PATH = "features/train_features.pt"
    TRAIN_LABELS_PATH = "features/train_labels.pt"
    VAL_FEATURES_PATH = "features/val_features.pt"
    VAL_LABELS_PATH = "features/val_labels.pt"

    batch_size = 256
    num_epochs = 100
    base_learning_rate = 1e-4 * (batch_size / 8)

    # Model parameters
    embedding_dim = 1536
    num_classes = 2

    # Regularization
    label_smoothing = 0.1

    # Seed
    seed = 123

class FeatureDataset(Dataset):
    def __init__(self, feature_path, label_path):
        # Load everything into RAM once
        print(f"Loading features from {feature_path} into RAM...")
        self.features = torch.load(feature_path)
        self.labels = torch.load(label_path).long() # Ensure labels are Long tensor
        print(f"Done. Loaded {len(self.features)} samples.")

    def __len__(self):
        # Return the total number of samples
        return len(self.features)

    def __getitem__(self, idx):
        # Simply index into the pre-loaded tensors
        return self.features[idx], self.labels[idx]

def get_classifier_head():
    """Creates the simple classifier head model."""
    classifier_head = nn.Sequential(
        nn.Linear(Config.embedding_dim * 2, 512), # Input is 1536 * 2 = 3072
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, Config.num_classes)
    ).to(Config.device)
    return classifier_head

def train_epoch_fast(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for features, labels in tqdm(dataloader, desc="Training"):
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        scheduler.step()

        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

def validate_fast(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in tqdm(dataloader, desc="Validation"):
            features, labels = features.to(device), labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc, all_preds, all_labels

def main():
    print(f"Using device: {Config.device}")
    torch.manual_seed(Config.seed)
    np.random.seed(Config.seed)

    train_feat_dataset = FeatureDataset(Config.TRAIN_FEATURES_PATH, Config.TRAIN_LABELS_PATH)
    val_feat_dataset = FeatureDataset(Config.VAL_FEATURES_PATH, Config.VAL_LABELS_PATH)

    print("Calculating class weights for sampler...")
    train_labels = train_feat_dataset.labels.numpy()
    class_counts = np.bincount(train_labels)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    samples_weight = class_weights[train_labels]

    sampler = WeightedRandomSampler(
        weights=samples_weight,
        num_samples=len(samples_weight),
        replacement=True
    )
    print("Sampler created.")

    train_loader = DataLoader(
        train_feat_dataset,
        batch_size=Config.batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_feat_dataset,
        batch_size=Config.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model = get_classifier_head()
    print(f"Classifier head created. Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    criterion = nn.CrossEntropyLoss(label_smoothing=Config.label_smoothing)

    optimizer = optim.Adam(model.parameters(), lr=Config.base_learning_rate)

    from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
    num_steps_per_epoch = len(train_loader)

    scheduler = CosineAnnealingWarmRestarts(
        optimizer,
        T_0=num_steps_per_epoch,
        eta_min=1e-6
    )


    best_val_acc = 0.0

    print("\nStarting FAST Training...")
    for epoch in range(Config.num_epochs):
        print(f"\nEpoch {epoch+1}/{Config.num_epochs}")

        train_loss, train_acc = train_epoch_fast(model, train_loader, criterion, optimizer, scheduler, Config.device)

        val_loss, val_acc, val_preds, val_labels = validate_fast(model, val_loader, criterion, Config.device)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_classifier_head.pth")
            print(f"Saved best model (val acc: {val_acc:.4f})")

    print("\nTraining Complete")

    print("Final Evaluation")
    model.load_state_dict(torch.load("best_classifier_head.pth"))
    _, val_acc, val_preds, val_labels = validate_fast(model, val_loader, criterion, Config.device)

    print(f"Best validation accuracy: {val_acc:.4f}")
    print("\nConfusion Matrix:")
    cm = confusion_matrix(val_labels, val_preds)
    print(cm)

    print("\nClassification Report:")

    class_names = ['1 (Non-mitotic)', '2 (Mitotic)']
    print(classification_report(val_labels, val_preds, target_names=class_names))

if __name__ == "__main__":
    main()